# Audio command classifier -- tuned multi-seed retrain (Colab GPU)

Continues `docs/plans/audio_eval_notebook_refactor_plan.md`. Local CPU experiments established:

- Two prior full retrains (`v4`: quarantine-cleaned dataset only, `v5`: + background-diversification) both **overfit** with a plain 60-epoch run (train acc -> ~100%, val acc noisy) and showed a confusing offline-vs-live-stream gap (`v5` improved every offline metric but *regressed* the live continuous-stream test vs. `v4`).
- A local augmentation ablation (25 epochs, 30% train subset, single seed) found **synthetic reverb is clearly harmful** (57% vs. 76% baseline val acc) and should be excluded. The other techniques (noise-mixing, speed-perturbation, gain/time-shift jitter) were directionally mixed but inconclusive at that budget -- exactly what a longer, multi-seed sweep is for.
- The dataset itself has a real domain gap: the original 6 classes are ~70% real human recordings, but the 5 movement classes added later (`forward`/`backward`/`left`/`right`/`go_grey`) are 100% synthetic TTS with a 3.3x smaller clip count -- motivating both augmentation (fake the missing real-world variation) and class weighting (fix the count imbalance).

This notebook runs the **real** tuning pass this was all leading up to: every augmentation preset x multiple seeds, with class weighting, weight decay, and early stopping, at a full epoch budget GPU compute makes affordable. It reuses `training/train_audio_command_classifier.py` unchanged (imported, not reimplemented) so the winning config can be reproduced locally afterward with the exact same code.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU -- Runtime > Change runtime type > GPU, then re-run this cell.')

In [ ]:
# soundfile + scipy usually need no system deps on Colab; torch/numpy are
# preinstalled. If soundfile import fails below, uncomment:
# !apt-get -qq install -y libsndfile1
!pip install -q soundfile

## Get the code + dataset

Built locally with `data_processing/prepare_colab_package.py`, which zips the training code and `data/synthetic+real_dataset_large/training_v2/` preserving the exact `ml_audio/...` relative paths -- so after extracting under `host_software/`, everything below runs unchanged from the local scripts, no path patching needed.

**Recommended: Google Drive** (the zip is ~700MB+; re-uploading it through the browser every session is slow and this survives runtime restarts). Upload `ml_audio_colab_package.zip` to your Drive once, then just mount + copy here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this to wherever you uploaded the zip in Drive.
DRIVE_ZIP_PATH = '/content/drive/MyDrive/ml_audio_colab_package.zip'

# Everything the sweep produces lives here -- survives a crash/disconnect/
# runtime recycle, since it's on Drive rather than the ephemeral Colab VM
# disk. See the "resumable sweep" section below for how this is used.
import os
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ml_audio_colab_output'
CHECKPOINTS_DIR = os.path.join(DRIVE_OUTPUT_DIR, 'checkpoints')
PROGRESS_PATH = os.path.join(DRIVE_OUTPUT_DIR, 'sweep_progress.json')
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
print('Sweep output directory:', DRIVE_OUTPUT_DIR)

In [ ]:
# Alternative to the Drive cell above: direct browser upload instead.
# from google.colab import files
# uploaded = files.upload()  # select ml_audio_colab_package.zip
# DRIVE_ZIP_PATH = next(iter(uploaded.keys()))

In [ ]:
import os, sys, zipfile

os.makedirs('host_software', exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP_PATH) as zf:
    zf.extractall('host_software')

HOST_SOFTWARE_DIR = os.path.abspath('host_software')
if HOST_SOFTWARE_DIR not in sys.path:
    sys.path.insert(0, HOST_SOFTWARE_DIR)

print(os.listdir(os.path.join(HOST_SOFTWARE_DIR, 'ml_audio')))

In [ ]:
from ml_audio.training.train_audio_command_classifier import (
    DEFAULT_DATASET_ROOT, OUTPUT_SEQUENCE_LENGTH,
    TrainableAudioCommandClassifier,
    discover_labels, build_waveform_cache, batch_spectrograms,
    compute_class_weights, train,
)
from ml_audio.evaluations.evaluate_audio_classifier import gather_labeled_files
from ml_audio.training.audio_augmentations import PRESETS
import numpy as np, torch, time, json

print('Dataset root:', DEFAULT_DATASET_ROOT)
print('Exists:', os.path.isdir(DEFAULT_DATASET_ROOT))
print('Augmentation presets:', list(PRESETS.keys()))

## Load the dataset once

Raw waveforms are cached (not spectrograms) so each sweep run can apply its own augmentation fresh per epoch -- see `train_audio_command_classifier.py`'s docstring for why. This is the expensive one-time cost; everything after this cell is fast.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

labels = discover_labels(DEFAULT_DATASET_ROOT)
label_to_idx = {l: i for i, l in enumerate(labels)}
print(f'{len(labels)} classes:', labels)

train_files = gather_labeled_files(os.path.join(DEFAULT_DATASET_ROOT, 'train'), labels)
val_files = gather_labeled_files(os.path.join(DEFAULT_DATASET_ROOT, 'val'), labels)
print('train:', len(train_files), ' val:', len(val_files))

t0 = time.time()
train_waveforms, train_targets = build_waveform_cache(train_files, label_to_idx)
val_waveforms, val_targets = build_waveform_cache(val_files, label_to_idx)
val_specs = batch_spectrograms(val_waveforms, device=device)
print(f'Loaded in {time.time() - t0:.1f}s')

sample_specs = batch_spectrograms(train_waveforms[:4000])
norm_mean = sample_specs.mean().reshape(1)
norm_var = sample_specs.var(unbiased=False).reshape(1)
print(f'norm stats: mean={norm_mean.item():.4f} var={norm_var.item():.4f}')

background_idx = label_to_idx['_background_']
noise_pool = train_waveforms[train_targets.numpy() == background_idx]
class_weights = compute_class_weights(train_targets, len(labels))
print('noise pool size:', len(noise_pool))
print('class weights:', {l: round(w, 2) for l, w in zip(labels, class_weights.tolist())})

## The sweep (resumable -- safe to leave running overnight)

Every augmentation preset except `none`-vs-baseline is worth comparing with class weighting always on (motivated by the real 3.3x count gap, not something we're trying to ablate) -- the open question from local testing is specifically *which augmentation*, so that's what varies. Reverb is excluded from `PRESETS` already (see plan doc). Adjust `SEEDS` down if you're compute/time constrained; 3 is enough to distinguish a real effect from single-run noise, which is exactly what the local ambiguity (v4 vs v5, and the inconclusive local ablation) was missing.

**Crash/disconnect resilience:** progress is written to `PROGRESS_PATH` on Drive after *every epoch* of every run (cheap -- a few KB of JSON), and a checkpoint is saved to `CHECKPOINTS_DIR` on Drive every time a run finds a new best val accuracy (also cheap -- this model is ~13.5K parameters, checkpoints are tens of KB). If Colab disconnects or the runtime dies mid-sweep:

1. Reconnect, re-run cells 1-9 (setup + dataset load -- the dataset itself isn't cached across sessions, so this ~2-10 min step does repeat).
2. Re-run this cell. Any run already marked `"done"` in `PROGRESS_PATH` is skipped instantly. A run that was `"in_progress"` when the crash happened restarts from epoch 1 (its optimizer state isn't checkpointed, only the best weights found so far) -- so a crash costs at most one run's progress (bounded by `PATIENCE` x wall-clock/epoch), never the whole sweep.

You can also inspect progress at any time, mid-sweep or after, by re-running cells 12-13 (the leaderboard) -- they read `PROGRESS_PATH` from Drive directly rather than relying on this cell's in-memory state, so they work correctly even in a fresh session.

In [ ]:
import json as _json


def load_progress():
    if os.path.exists(PROGRESS_PATH):
        with open(PROGRESS_PATH) as f:
            return _json.load(f)
    return {}


def save_progress(progress):
    # Write-then-replace rather than an in-place write, so a crash mid-write
    # can't leave a truncated/corrupt progress file behind.
    tmp_path = PROGRESS_PATH + '.tmp'
    with open(tmp_path, 'w') as f:
        _json.dump(progress, f, indent=2)
    os.replace(tmp_path, PROGRESS_PATH)


progress = load_progress()
if progress:
    print(f'Resuming -- {len(progress)} previously recorded run(s):')
    for k, v in progress.items():
        acc = v.get('best_val_acc', 0.0)
        print(f"  {k}: {v['status']}  best_val_acc={acc:.3%}")
else:
    print('Starting fresh sweep (no existing progress found on Drive).')

SWEEP_CONFIGS = ['none', 'light', 'speed', 'noise', 'combined']
SEEDS = [0, 1, 2]
EPOCHS = 150
PATIENCE = 20
BATCH_SIZE = 128  # larger than the CPU default (64) -- cheap on GPU, faster convergence per wall-clock
LR = 1e-3
WEIGHT_DECAY = 1e-4


def make_on_epoch_end(key, config_name, seed, checkpoint_path):
    # Factory (not a plain closure over the loop variables) so each run's
    # callback captures its own key/config_name/seed/checkpoint_path even
    # though the loop keeps reassigning those names -- avoids the classic
    # late-binding-closure bug.
    running_best = {'val_acc': progress.get(key, {}).get('best_val_acc', 0.0)}

    def on_epoch_end(epoch, val_acc, is_best, best_state):
        if is_best:
            running_best['val_acc'] = val_acc
        progress[key] = {
            'status': 'in_progress', 'config': config_name, 'seed': seed,
            'epoch': epoch, 'best_val_acc': running_best['val_acc'],
        }
        save_progress(progress)
        if is_best:
            # Native TrainableAudioCommandClassifier state dict, NOT the
            # AudioCommandClassifier-compatible export format -- that
            # conversion happens once, for the overall winner, in the
            # "save the best checkpoint" cell below.
            torch.save(best_state, checkpoint_path)

    return on_epoch_end


for config_name in SWEEP_CONFIGS:
    augmentation = PRESETS[config_name]
    for seed in SEEDS:
        key = f'{config_name}_seed{seed}'

        if progress.get(key, {}).get('status') == 'done':
            print(f"\n=== {key}: already done (val_acc={progress[key]['best_val_acc']:.3%}), skipping ===")
            continue

        print(f'\n=== {key} ===')
        torch.manual_seed(seed)
        model = TrainableAudioCommandClassifier(num_classes=len(labels))
        model.set_input_normalization(norm_mean, norm_var)

        checkpoint_path = os.path.join(CHECKPOINTS_DIR, f'{key}.pth')
        on_epoch_end = make_on_epoch_end(key, config_name, seed, checkpoint_path)

        t0 = time.time()
        try:
            best_state, best_val_acc, val_accs = train(
                model, train_waveforms, train_targets, val_specs, val_targets,
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
                augmentation=augmentation, noise_pool=noise_pool,
                class_weights=class_weights, patience=PATIENCE, seed=seed, device=device,
                on_epoch_end=on_epoch_end,
            )
        except Exception as e:
            # One bad run (OOM, a transient Drive I/O hiccup, etc.) shouldn't
            # take the rest of the overnight sweep down with it.
            print(f'  RUN FAILED: {e!r}')
            progress[key] = {'status': 'failed', 'config': config_name, 'seed': seed, 'error': repr(e)}
            save_progress(progress)
            continue

        elapsed = time.time() - t0
        progress[key] = {
            'status': 'done', 'config': config_name, 'seed': seed,
            'best_val_acc': best_val_acc, 'epochs_run': len(val_accs), 'seconds': elapsed,
        }
        save_progress(progress)
        print(f'{key}: best_val_acc={best_val_acc:.3%} epochs_run={len(val_accs)} ({elapsed:.0f}s) -- saved to Drive')

print('\nSweep cell finished (all runs done or already were).')

## Compare configs (mean +/- std across seeds)

This is the number the local single-seed ablation couldn't give us -- whether a config's edge is real or noise. Reads `PROGRESS_PATH` fresh from Drive, so this works whether the sweep cell just finished, is still running (partial leaderboard), or this is a new session after a crash.

In [ ]:
import statistics

progress = load_progress()  # reload -- may be a fresh session after a crash

by_config = {}
for key, r in progress.items():
    if r.get('status') != 'done':
        continue
    by_config.setdefault(r['config'], []).append(r['best_val_acc'])

summary = []
for config_name, accs in by_config.items():
    mean = statistics.mean(accs)
    std = statistics.stdev(accs) if len(accs) > 1 else 0.0
    summary.append((config_name, mean, std, accs))
summary.sort(key=lambda x: -x[1])

print(f"{'config':16s} {'mean':>8s} {'std':>7s}  per-seed")
for config_name, mean, std, accs in summary:
    accs_str = ', '.join(f'{a:.3%}' for a in accs)
    print(f'{config_name:16s} {mean:.3%} {std:.3%}  [{accs_str}]  ({len(accs)}/{len(SEEDS)} seeds done)')

not_done = {k: v['status'] for k, v in progress.items() if v.get('status') != 'done'}
if not_done:
    print(f'\nNote: sweep not fully finished -- this leaderboard is partial. Not done: {not_done}')

if summary:
    print(f'\nWinning config so far by mean val acc: {summary[0][0]}')

## Save the best checkpoint

Picks the single best **completed** run (highest val acc among runs marked `"done"`) and exports it in the same `AudioCommandClassifier`-compatible format `train_audio_command_classifier.py` produces locally -- drop-in for `evaluate_audio_classifier.py` / `audio_receiver_pytorch.py`, no changes needed. Reads the winning run's weights from `CHECKPOINTS_DIR` on Drive (not from this session's memory), so this cell works standalone even in a brand new session, as long as the sweep has produced at least one completed run.

In [ ]:
progress = load_progress()
done_runs = {k: v for k, v in progress.items() if v.get('status') == 'done'}
if not done_runs:
    raise RuntimeError('No completed runs yet -- run the sweep cell first (or wait for it to finish).')

best_key = max(done_runs, key=lambda k: done_runs[k]['best_val_acc'])
best_run = done_runs[best_key]
print(f'Best completed run: {best_key}  val_acc={best_run["best_val_acc"]:.3%}')

best_checkpoint_path = os.path.join(CHECKPOINTS_DIR, f'{best_key}.pth')
state_dict = torch.load(best_checkpoint_path, map_location='cpu')

final_model = TrainableAudioCommandClassifier(num_classes=len(labels))
final_model.load_state_dict(state_dict)
final_model.eval()

OUT_DIR = 'colab_output'
os.makedirs(OUT_DIR, exist_ok=True)
checkpoint_path = os.path.join(OUT_DIR, 'audio_command_classifier_state_dict_v6.pth')
torch.save(final_model.to_inference_state_dict(), checkpoint_path)
with open(os.path.join(OUT_DIR, 'labels.json'), 'w') as f:
    json.dump(labels, f, indent=2)
with open(os.path.join(OUT_DIR, 'sweep_summary.json'), 'w') as f:
    json.dump(progress, f, indent=2)

print('Saved to', OUT_DIR)
print('(per-run checkpoints + full progress log remain on Drive at', DRIVE_OUTPUT_DIR, ')')
print(os.listdir(OUT_DIR))

## Download the result

Copy `audio_command_classifier_state_dict_v6.pth` + `labels.json` into `host_software/ml_audio/models/pytorch_v6/` locally, then re-run `evaluations/evaluate_audio_classifier.py` and `evaluations/evaluate_live_receiver_stream.py` exactly as for v4/v5 -- offline accuracy AND the live-stream number both matter (recall v5 improved offline metrics but regressed live-stream vs. v4; don't declare victory on offline accuracy alone).

Note the winning checkpoint is already durably saved on Drive (`DRIVE_OUTPUT_DIR`) regardless of whether you run this cell -- this is just a convenience download of the final packaged result.

In [ ]:
import shutil
shutil.make_archive('colab_output', 'zip', OUT_DIR)

from google.colab import files
files.download('colab_output.zip')
# Or, to persist a copy of just the final zip to Drive as well (the
# checkpoints/progress are already there per-run, this is just the bundled
# final result):
# shutil.copy('colab_output.zip', '/content/drive/MyDrive/colab_output.zip')